# 조달청 OpenAPI 데이터 수집

위에서 아래로 모든 셀 실행하면 **4개 CSV 파일이 노트북 폴더에 자동 생성**됩니다:

- `bid_notices.csv` — 입찰공고
- `opening.csv` — 개찰결과 (1순위 투찰자)
- `award.csv` — 최종 낙찰자
- `prepar.csv` — 복수예가 15개

## 사전 준비

1. **공공데이터포털** ([data.go.kr](https://www.data.go.kr/)) 활용신청 — 두 개 서비스:
   - `조달청_나라장터 입찰공고정보서비스`
   - `조달청_나라장터 낙찰정보서비스`
2. 승인 후 **Decoding 키** 발급
3. 아래 셀에서 `SERVICE_KEY`에 키 입력 (또는 환경변수 `G2B_SERVICE_KEY` 사용)

## 호출하는 API

| 단계 | operationId | 무엇을 가져오나 |
|---|---|---|
| ① 공고 게시 | `getBidPblancListInfoThngPPSSrch` | 입찰공고 — 어떤 발주가 떴나 |
| ② 개찰 결과 | `getOpengResultListInfoThng` | 1순위·유찰·재시담 |
| ③ 최종 낙찰 | `getScsbidListSttusThng` | 진짜 계약한 회사 |
| ④ 복수예가 | `getOpengResultListInfoThngPreparPcDetail` | 예비가격 15개 분포 |

## 0. 셋업 — ServiceKey + 공통 헬퍼

In [ ]:
import os, re
import requests
import pandas as pd
import xml.etree.ElementTree as ET

# 1) ServiceKey — 환경변수 또는 직접 입력
# SERVICE_KEY = os.environ.get("G2B_SERVICE_KEY") or "여기에_본인_Decoding_키"
SERVICE_KEY = "f4553816ce5af460573db642542fe3dbd83d3212d088e8415f4983832330599e"
assert SERVICE_KEY != "여기에_본인_Decoding_키", "ServiceKey를 입력하거나 환경변수 G2B_SERVICE_KEY 설정"

# 2) 엔드포인트
BASE_URL    = "http://apis.data.go.kr/1230000"
BID_SVC     = "ad/BidPublicInfoService"   # 14번
SCSBID_SVC  = "as/ScsbidInfoService"      # 1·5·9번
TIMEOUT     = (10, 60)

# 3) 한국환경공단 10개 권역 코드
ENV_CORP_DMINSTT = {
    "B552584": "한국환경공단 (본사)",
    "Z008653": "수도권서부환경본부",
    "Z004865": "수도권동부환경본부",
    "Z018993": "충청권환경본부",
    "Z004863": "부산울산경남환경본부",
    "Z003477": "대구경북환경본부",
    "Z004864": "광주전남제주환경본부",
    "Z042470": "강원환경본부",
    "D266078": "국가물산업클러스터사업단",
    "Z042433": "전북환경본부",
}
print("✅ 셋업 완료")

In [ ]:
# 공통 헬퍼 — API 호출 / 정규화 / 파서

def call_api(operation_path: str, params: dict) -> ET.Element:
    """G2B API 호출 → XML root.
    
    에러 분기:
      - HTTP 4xx/5xx       → raise_for_status()
      - 게이트웨이 에러     → root tag = OpenAPI_ServiceResponse
      - 정상 응답 에러     → resultCode != '00'
    """
    url = f"{BASE_URL}/{operation_path}"
    r = requests.get(url, params={"ServiceKey": SERVICE_KEY, "type": "xml", **params}, timeout=TIMEOUT)
    r.raise_for_status()
    root = ET.fromstring(r.text)
    if root.tag == "OpenAPI_ServiceResponse":
        raise RuntimeError(f"Gateway error: {root.findtext('.//returnAuthMsg')}")
    if root.findtext(".//resultCode") != "00":
        raise RuntimeError(f"resultCode={root.findtext('.//resultCode')}, msg={root.findtext('.//resultMsg')}")
    return root

def normalize_brn(s):
    """BRN → 하이픈 없는 10자리."""
    if not s: return None
    digits = re.sub(r"\D", "", s)
    return digits if len(digits) == 10 else None

def to_int(s):
    try: return int(s.strip()) if s and s.strip() else None
    except: return None

def to_float(s):
    try: return float(s.strip()) if s and s.strip() else None
    except: return None

def parse_openg_corp_info(s):
    """5번 API의 opengCorpInfo caret(^) 묶음 필드 파싱.
    
    case ∈ {single, multiple, negotiation, empty, malformed}
    """
    if not s or not s.strip():
        return {"case": "empty", "winner_name": None, "brn": None, "ceo_name": None, "bid_amount": None, "bid_rate": None}
    tokens = s.split("^")
    if tokens[0].startswith("낙찰예정자 다수"):
        return {"case": "multiple", "winner_name": None, "brn": None, "ceo_name": None,
                "bid_amount": to_int(tokens[-2]) if len(tokens) >= 2 else None,
                "bid_rate": to_float(tokens[-1]) if len(tokens) >= 1 else None}
    if len(tokens) >= 5:
        has_amt = bool(tokens[3].strip()) and bool(tokens[4].strip())
        return {"case": "single" if has_amt else "negotiation",
                "winner_name": tokens[0] or None, "brn": normalize_brn(tokens[1]),
                "ceo_name": tokens[2] or None,
                "bid_amount": to_int(tokens[3]), "bid_rate": to_float(tokens[4])}
    return {"case": "malformed", "winner_name": None, "brn": None, "ceo_name": None, "bid_amount": None, "bid_rate": None}

print("✅ 헬퍼 준비 완료")

## ① 입찰공고 — `getBidPblancListInfoThngPPSSrch` (14번)

**"어떤 발주가 떴나"** — 입찰 시작 단계.

환경공단 본사(B552584)의 2024년 4월 1주일치 공고를 조회.

In [ ]:
def fetch_bid_notice(begin_dt, end_dt, dminstt_cd=None, page_size=100):
    """입찰공고 조회.
    
    Args:
        begin_dt, end_dt: YYYYMMDDHHMM (예: '202404010000') — 30일 이내
        dminstt_cd:       수요기관 코드 (없으면 전체)
    """
    params = {"inqryDiv": "1", "inqryBgnDt": begin_dt, "inqryEndDt": end_dt,
              "pageNo": "1", "numOfRows": str(page_size)}
    if dminstt_cd: params["dminsttCd"] = dminstt_cd
    root = call_api(f"{BID_SVC}/getBidPblancListInfoThngPPSSrch", params)
    items = []
    for it in root.findall(".//item"):
        items.append({
            "bid_ntce_no":    it.findtext("bidNtceNo"),
            "bid_ntce_ord":   it.findtext("bidNtceOrd"),
            "bid_ntce_nm":    it.findtext("bidNtceNm"),
            "dminstt_cd":     it.findtext("dminsttCd"),
            "dminstt_nm":     it.findtext("dminsttNm"),
            "ntce_instt_nm":  it.findtext("ntceInsttNm"),
            "dtil_prdct_no":  it.findtext("dtilPrdctClsfcNo"),
            "dtil_prdct_nm":  it.findtext("dtilPrdctClsfcNoNm"),
            "presmpt_prce":   to_int(it.findtext("presmptPrce")),
            "asign_bdgt_amt": to_int(it.findtext("asignBdgtAmt")),
            "bid_ntce_dt":    it.findtext("bidNtceDt"),
            "openg_dt":       it.findtext("opengDt"),
            "bid_clse_dt":    it.findtext("bidClseDt"),
        })
    total = to_int(root.findtext(".//totalCount")) or 0
    return items, total

items, total = fetch_bid_notice(
    begin_dt="202404010000",
    end_dt="202404072359",
    dminstt_cd="B552584",  # 환경공단 본사
)
df_bid = pd.DataFrame(items)
print(f"총 {total}건")
df_bid

## ② 개찰결과 — `getOpengResultListInfoThng` (5번)

**"입찰 결과 — 1순위·유찰·재시담·재입찰"** — 입찰 직후 시점.

특정 공고의 개찰 결과 조회. 1순위 투찰자 BRN과 가격, 유찰 여부 등을 알 수 있음.

In [ ]:
def fetch_opening_result(bid_ntce_no):
    """특정 공고의 개찰결과.
    
    progrs_div_cd_nm 가능 값: '개찰완료' / '유찰' / '재입찰' / '재시담'
    """
    params = {"inqryDiv": "4", "bidNtceNo": bid_ntce_no, "pageNo": "1", "numOfRows": "100"}
    root = call_api(f"{SCSBID_SVC}/getOpengResultListInfoThng", params)
    out = []
    for it in root.findall(".//item"):
        corp = parse_openg_corp_info(it.findtext("opengCorpInfo"))
        out.append({
            "bid_ntce_no":      it.findtext("bidNtceNo"),
            "bid_ntce_ord":     it.findtext("bidNtceOrd"),
            "bid_clsfc_no":     it.findtext("bidClsfcNo"),
            "rbid_no":          it.findtext("rbidNo"),
            "openg_dt":         it.findtext("opengDt"),
            "prtcpt_cnum":      to_int(it.findtext("prtcptCnum")),
            "progrs_div_cd_nm": it.findtext("progrsDivCdNm"),
            "rsrvtn_prce_yn":   it.findtext("rsrvtnPrceFileExistnceYn"),
            "dminstt_nm":       it.findtext("dminsttNm"),
            "case":             corp["case"],
            "winner_name":      corp.get("winner_name"),
            "winner_brn":       corp.get("brn"),
            "bid_amount":       corp.get("bid_amount"),
            "bid_rate":         corp.get("bid_rate"),
        })
    return out

# 위 14번 결과의 첫 공고 사용
sample_bid = df_bid["bid_ntce_no"].iloc[0]
df_open = pd.DataFrame(fetch_opening_result(sample_bid))
print(f"공고: {sample_bid}")
df_open

## ③ 최종 낙찰자 — `getScsbidListSttusThng` (1번)

**"진짜 계약한 회사"** — 적격심사·협상 끝난 후 확정.

유찰된 공고는 row가 없음. 5번과 비교해 차이가 있으면 협상으로 차순위가 낙찰받은 케이스.

In [ ]:
def fetch_award(bid_ntce_no):
    """특정 공고의 최종 낙찰자."""
    params = {"inqryDiv": "4", "bidNtceNo": bid_ntce_no, "pageNo": "1", "numOfRows": "100"}
    root = call_api(f"{SCSBID_SVC}/getScsbidListSttusThng", params)
    out = []
    for it in root.findall(".//item"):
        out.append({
            "bid_ntce_no":     it.findtext("bidNtceNo"),
            "bid_ntce_nm":     it.findtext("bidNtceNm"),
            "bidwinnr_nm":     it.findtext("bidwinnrNm"),
            "bidwinnr_brn":    normalize_brn(it.findtext("bidwinnrBizno")),
            "bidwinnr_ceo":    it.findtext("bidwinnrCeoNm"),
            "bidwinnr_adrs":   it.findtext("bidwinnrAdrs"),
            "sucsfbid_amt":    to_int(it.findtext("sucsfbidAmt")),
            "sucsfbid_rate":   to_float(it.findtext("sucsfbidRate")),
            "prtcpt_cnum":     to_int(it.findtext("prtcptCnum")),
            "fnl_sucsf_date":  it.findtext("fnlSucsfDate"),
            "dminstt_nm":      it.findtext("dminsttNm"),
        })
    return out

df_award = pd.DataFrame(fetch_award(sample_bid))
print(f"공고: {sample_bid}")
df_award

## ④ 복수예가 상세 — `getOpengResultListInfoThngPreparPcDetail` (9번)

**"발주처가 미리 만든 예비가격 15개의 분포"**

**복수예가 메커니즘:**
1. 발주처가 입찰 전 예비가격 15개를 미리 만들어둠
2. 입찰자들이 가격 부름
3. 그 중 4개를 추첨해서 평균 = **기초가격**
4. 기초가격에 가장 가까운 가격을 부른 업체가 낙찰

5번에서 `rsrvtn_prce_yn == 'Y'` 인 공고만 호출 의미 있음.

⚠️ `inqryDiv` 의미가 다른 API와 다름 — 9번에서는 `2`=입찰공고번호 (1·5번은 `4`)

In [ ]:
def fetch_prepar_price(bid_ntce_no):
    """예비가격 15개 디테일."""
    params = {"inqryDiv": "2", "bidNtceNo": bid_ntce_no, "pageNo": "1", "numOfRows": "30"}
    root = call_api(f"{SCSBID_SVC}/getOpengResultListInfoThngPreparPcDetail", params)
    out = []
    for it in root.findall(".//item"):
        out.append({
            "bid_ntce_no":         it.findtext("bidNtceNo"),
            "bid_ntce_nm":         it.findtext("bidNtceNm"),
            "rsrvtn_prce_sno":     to_int(it.findtext("compnoRsrvtnPrceSno")),
            "tot_rsrvtn_prce_num": to_int(it.findtext("totRsrvtnPrceNum")),
            "plnprc":              to_int(it.findtext("plnprc")),
            "bssamt":              to_int(it.findtext("bssamt")),
            "bsis_plnprc":         to_int(it.findtext("bsisPlnprc")),
            "drwt_yn":             it.findtext("drwtYn"),
            "drwt_num":            to_int(it.findtext("drwtNum")),
        })
    return out

df_prepar = pd.DataFrame(fetch_prepar_price(sample_bid))
print(f"공고: {sample_bid}")
print(f"기초가격: {df_prepar['bssamt'].iloc[0]:,}원" if not df_prepar.empty else "")
df_prepar

## ⑤ 1순위 vs 최종낙찰자 비교 — 협상 거래 잡기

5번(개찰)의 1순위와 1번(낙찰자)의 최종이 **다르면** = 협상으로 차순위가 받은 것.
(덤핑 / 적격심사 탈락 / 계약 거절 등)

In [ ]:
if not df_open.empty and not df_award.empty:
    top1_brn  = df_open["winner_brn"].iloc[0]
    final_brn = df_award["bidwinnr_brn"].iloc[0]
    if top1_brn == final_brn:
        print(f"✅ 1순위 == 최종: {top1_brn} (정상 낙찰)")
    else:
        print(f"⚠️ 1순위({top1_brn}) ≠ 최종({final_brn})")
        print("  → 협상으로 차순위 낙찰 (덤핑·적격탈락 의심)")
else:
    print("비교 불가 (개찰/낙찰 둘 중 하나 비어있음)")

## ⑥ CSV로 저장하기 — Excel에서 바로 열림

`utf-8-sig` 인코딩이 BOM을 포함해서 Excel이 한글 깨짐 없이 열어줌.

In [ ]:
df_bid.to_csv("bid_notices.csv", index=False, encoding="utf-8-sig")
df_open.to_csv("opening.csv", index=False, encoding="utf-8-sig")
df_award.to_csv("award.csv", index=False, encoding="utf-8-sig")
df_prepar.to_csv("prepar.csv", index=False, encoding="utf-8-sig")
print("✅ 4개 CSV 저장 완료 (현재 폴더)")

## ⑦ 1개월치 환경공단 공고 한꺼번에 — 백필 패턴

G2B는 `inqryBgnDt~inqryEndDt`가 **30일 초과면 에러**.
더 긴 기간을 보려면 윈도우로 쪼개서 호출.

In [ ]:
from datetime import date, timedelta

def month_windows(start, end, days=30):
    cur = start
    while cur < end:
        nxt = min(cur + timedelta(days=days), end)
        yield (cur.strftime("%Y%m%d") + "0000", nxt.strftime("%Y%m%d") + "2359")
        cur = nxt + timedelta(days=1)

# 2024-04 ~ 2024-06 환경공단 본사 공고 전부
all_items = []
for begin, end in month_windows(date(2024, 4, 1), date(2024, 6, 30)):
    items, total = fetch_bid_notice(begin, end, dminstt_cd="B552584")
    print(f"  · {begin} ~ {end}: {total}건")
    all_items.extend(items)

df_3mo = pd.DataFrame(all_items)
print(f"\n총 {len(df_3mo)}건")
df_3mo[["bid_ntce_no", "bid_ntce_nm", "presmpt_prce", "openg_dt"]].head()